# CertVIC main-200 — Diffusion edits (Kaggle T4×2)

Generates the **168 photorealistic single-factor edits** of the pilot. Two
workers run in parallel — **GPU 0 → shard 0 (81)**, **GPU 1 → shard 1 (87)**.

**Settings:** Accelerator = **GPU T4 ×2**. **Internet = On for run 1** (to fetch
the open SD-2-inpaint weights), or **Off** if you pre-cached them as a dataset
(`00_precache_weights.ipynb`). The weights cell finds a mounted snapshot if present,
else downloads the free model automatically.

**This notebook does NOT run VLM inference.** VLM eval is a separate notebook,
blocked until quality + detectability (AUC < 0.80) + human review + item
certificates pass.

**Estimated runtime (T4×2):** model load ~1–2 min; generation ~10–15 min
(~84 edits/GPU at ~6–8 s each, in parallel). Single-T4 would be ~20–25 min.

## 1. Attach inputs (Add Data)
- **certvic** — this bundle (`certvic_kaggle_main200_bundle.zip`); Kaggle mounts it at
  `/kaggle/input/<your-slug>/` and the setup cell **auto-detects** it (slug can be anything).
- **ADE20K** — a dataset containing `ADEChallengeData2016/{images,annotations}/...`.
- **weights (recommended)** — click **+ Add Input → Models** and add the non-gated
  mirror **`kaggle.com/refs/hf-model/stable-diffusion-v1-5/stable-diffusion-inpainting`**
  (no HF token, runs **Internet OFF**). The weights cell auto-detects its
  `model_index.json`. *Alternatives:* skip it and the cell downloads the same model
  on first run (Internet ON); or set `HF_TOKEN` + `SD_MODEL_ID` for a gated repo
  (a **401** just means the repo you picked is gated).

Then set `ADE20K_ROOT` (next cell auto-detects if you forget).

In [ ]:
# Dependency check (Kaggle GPU images usually ship these). If MISSING, enable
# Internet briefly and run:  %pip install -q diffusers transformers accelerate safetensors
import importlib.util
for m in ("torch", "diffusers", "huggingface_hub", "PIL", "numpy"):
    print(m, "OK" if importlib.util.find_spec(m) else "MISSING -> %pip install")

In [ ]:
# --- mounts + path remap (no hardcoded private paths) ---
import os, sys, json, glob
from pathlib import Path

# Find the bundle wherever Kaggle mounted it (the dataset slug is derived from its
# title, not necessarily "certvic"; an un-extracted .zip is handled too).
def find_certvic():
    hits = glob.glob("/kaggle/input/**/main_real_200/gpu_shards/pilot_edit_plan_shard0_of_2.jsonl", recursive=True)
    if hits:
        return hits[0].split("/data/results/")[0]
    zips = glob.glob("/kaggle/input/**/certvic_kaggle_main200_bundle.zip", recursive=True)
    if zips:
        import zipfile
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/kaggle/working/certvic_bundle")
        return "/kaggle/working/certvic_bundle"
    raise FileNotFoundError("Attach the certvic bundle dataset (must contain data/results/main_real_200/...).")
CERTVIC = find_certvic()
print("CERTVIC:", CERTVIC)

# Weights resolution (best first): a mounted Kaggle snapshot needs no HF call at all.
# Default download id is a NON-GATED inpaint mirror so no token is required; override
# with env SD_MODEL_ID, and pass HF_TOKEN (Kaggle Add-ons > Secrets) for gated repos.
SD_MODEL_ID = os.environ.get("SD_MODEL_ID", "stable-diffusion-v1-5/stable-diffusion-inpainting")
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

def resolve_sd_weights():
    explicit = os.environ.get("SD_WEIGHTS")
    if explicit and os.path.exists(explicit):
        return explicit                                  # exact mount path, if you set SD_WEIGHTS
    hits = glob.glob("/kaggle/input/**/model_index.json", recursive=True)
    if hits:
        return os.path.dirname(hits[0])                  # any mounted diffusers snapshot (e.g. the Kaggle model)
    cached = "/kaggle/working/sd-inpaint"
    if os.path.exists(cached + "/model_index.json"):
        return cached                                    # already fetched this session
    from huggingface_hub import snapshot_download
    try:
        return snapshot_download(SD_MODEL_ID, local_dir=cached, token=HF_TOKEN, ignore_patterns=["*.ckpt"])
    except Exception as e:
        raise RuntimeError(
            f"No mounted weights and download of '{SD_MODEL_ID}' failed ({type(e).__name__}). Do ANY one: "
            "(0) BEST - click '+ Add Input' > Models and add the non-gated Kaggle mirror "
            "kaggle.com/refs/hf-model/stable-diffusion-v1-5/stable-diffusion-inpainting "
            "(no token, runs Internet OFF); this cell then auto-detects it. "
            "(1) Or set env SD_WEIGHTS to its exact /kaggle/input/... path. "
            "(2) Or for a gated repo, set HF_TOKEN (Add-ons > Secrets) + SD_MODEL_ID.") from e

WEIGHTS = resolve_sd_weights()
print("WEIGHTS:", WEIGHTS)

ADE20K_ROOT = os.environ.get("ADE20K_ROOT") or next(
    iter(glob.glob("/kaggle/input/**/ADEChallengeData2016", recursive=True)), None)
assert ADE20K_ROOT, "Attach an ADE20K dataset and set ADE20K_ROOT."
os.environ["ADE20K_ROOT"] = ADE20K_ROOT
sys.path.insert(0, CERTVIC)

WORK = Path("/kaggle/working"); (WORK / "edits").mkdir(parents=True, exist_ok=True)
shard_src = f"{CERTVIC}/data/results/main_real_200/gpu_shards"
# Detect the baked-in prefix (the __ADE20K_ROOT__ token in the bundle) and swap for the mount:
_r0 = json.loads(open(f"{shard_src}/pilot_edit_plan_shard0_of_2.jsonl").readline())
LOCAL_ROOT = os.environ.get("CERTVIC_LOCAL_ADE20K_ROOT") or _r0["image_path"].split("/images/")[0]

def remap(src, dst):
    rows = [json.loads(l) for l in open(src) if l.strip()]
    for r in rows:
        for k in ("image_path", "mask_path", "original_image_path", "annotation_path"):
            if r.get(k):
                r[k] = r[k].replace(LOCAL_ROOT, ADE20K_ROOT)
    open(dst, "w").writelines(json.dumps(r) + "\n" for r in rows)
    return len(rows)

for i in (0, 1):
    n = remap(f"{shard_src}/pilot_edit_plan_shard{i}_of_2.jsonl", str(WORK / f"plan_shard{i}.jsonl"))
    print(f"shard {i}: {n} edits remapped  {LOCAL_ROOT} -> {ADE20K_ROOT}")

In [ ]:
%%writefile /kaggle/working/engine_patch.py
# Real diffusion-inpaint engine, patched onto certvic.edit.engines._diffusers_inpaint.
# Tune _prompt_for / steps to pass the CPU detectability gate (AUC < 0.80).
import os, numpy as np, torch
from PIL import Image, ImageFilter
from diffusers import StableDiffusionInpaintPipeline
import certvic.edit.engines as engines

WEIGHTS = os.environ["WEIGHTS"]
_PIPE = None
def _pipe():
    global _PIPE
    if _PIPE is None:
        try:                                            # prefer the smaller fp16 variant
            p = StableDiffusionInpaintPipeline.from_pretrained(WEIGHTS, torch_dtype=torch.float16, variant="fp16", safety_checker=None)
        except Exception:                                # else default precision (also handles HF-id download)
            p = StableDiffusionInpaintPipeline.from_pretrained(WEIGHTS, torch_dtype=torch.float16, safety_checker=None)
        _PIPE = p.to("cuda"); _PIPE.set_progress_bar_config(disable=True)
    return _PIPE

def _prompt_for(plan):
    label = plan.get("label_name") or "object"
    et = plan.get("edit_type")
    if et == "remove":             return (f"empty background where the {label} was, photorealistic, consistent lighting", f"the {label}, object, artifacts")
    if et == "occlude":            return (f"a plain cardboard box partially covering the {label}, photorealistic", "")
    if et == "displace":           return (f"empty background, the {label} removed, photorealistic", "")
    if et == "control_irrelevant": return (f"the same scene with a repainted wall, the {label} unchanged", "")
    return ("photorealistic edited region", "")

def real_diffusers_inpaint(image, mask, plan, rng, seed):
    pipe = _pipe()
    exact = (np.asarray(mask) > 0).astype("uint8") * 255
    m = Image.fromarray(exact).filter(ImageFilter.MaxFilter(7))   # dilate for the inpaint only
    W, H = image.size
    base = image.convert("RGB").resize((512, 512)); mres = m.resize((512, 512))
    prompt, negative = _prompt_for(plan)
    g = torch.Generator(device="cuda").manual_seed(seed)
    out = pipe(prompt=prompt, negative_prompt=negative or None, image=base, mask_image=mres,
               num_inference_steps=30, guidance_scale=7.5, generator=g).images[0].resize((W, H))
    # Composite: keep ORIGINAL pixels OUTSIDE the object mask -> a true single-factor edit.
    # (Without this the VAE round-trip changes the whole image and fails the single-factor gate.)
    edited = Image.composite(out, image.convert("RGB"), Image.fromarray(exact))
    return edited, {"operation": "diffusers_inpaint_composited", "model": WEIGHTS,
                    "prompt": prompt, "steps": 30, "guidance_scale": 7.5, "seed": seed}

engines._diffusers_inpaint = real_diffusers_inpaint
print("patched engines._diffusers_inpaint")

In [ ]:
%%writefile /kaggle/working/worker.py
# One worker = one shard on one GPU. Reuses certvic.batch_generate (resume, dedup,
# quality gates, replay metadata, manifest schema) — only the engine is new.
import os, sys, json
sys.path.insert(0, os.environ["CERTVIC"])
exec(open("/kaggle/working/engine_patch.py").read())
import certvic.edit.engines as engines
shard = int(os.environ["SHARD"])
summary = engines.batch_generate(
    edit_plan_path=f"/kaggle/working/plan_shard{shard}.jsonl",
    out_dir=f"/kaggle/working/edits/shard{shard}",
    out_manifest=f"/kaggle/working/generated_shard{shard}.jsonl",
    rejected_out=f"/kaggle/working/rejected_shard{shard}.jsonl",
    summary_out=f"/kaggle/working/gen_summary_shard{shard}.json",
    engine="diffusers_inpaint_optional", max_items=1000, seed=0, resume=True, fail_fast=False)
print("shard", shard, json.dumps(summary))

In [ ]:
# --- launch GPU0 (shard0) + GPU1 (shard1) in parallel, with LIVE progress ---
# Worker output goes to log files; this cell polls the edited-PNG count every 20s so
# you can SEE it working (no output for ~1-2 min at first = model loading, not a hang).
import os, sys, time, glob, subprocess
EXPECT = {s: sum(1 for _ in open(f"/kaggle/working/plan_shard{s}.jsonl")) for s in (0, 1)}
def launch(gpu, shard):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "SHARD": str(shard),
           "CERTVIC": CERTVIC, "WEIGHTS": WEIGHTS, "ADE20K_ROOT": ADE20K_ROOT, "PYTHONUNBUFFERED": "1"}
    return subprocess.Popen([sys.executable, "-u", "/kaggle/working/worker.py"], env=env,
                            stdout=open(f"/kaggle/working/log_shard{shard}.txt", "w"),
                            stderr=subprocess.STDOUT)

procs = {0: launch(0, 0), 1: launch(1, 1)}       # GPU0 -> shard0, GPU1 -> shard1
t0 = time.time()
print(f"launched: GPU0={EXPECT[0]} edits, GPU1={EXPECT[1]} edits | first PNGs after ~1-2 min model load", flush=True)
while any(p.poll() is None for p in procs.values()):
    time.sleep(20)
    line = []
    for s in (0, 1):
        n = len(glob.glob(f"/kaggle/working/edits/shard{s}/*.png"))
        line.append(f"GPU{s} {n}/{EXPECT[s]} {'run' if procs[s].poll() is None else 'done'}")
    print(f"[{int(time.time()-t0):4d}s] " + " | ".join(line), flush=True)
for s in (0, 1):
    print(f"--- shard{s} log tail ---\n" + open(f"/kaggle/working/log_shard{s}.txt").read()[-600:])

In [ ]:
# --- merge shards + package edits + MANIFEST + summaries into ONE zip ---
import json, os, glob, zipfile
merged = []
for s in (0, 1):
    merged += [json.loads(l) for l in open(f"/kaggle/working/generated_shard{s}.jsonl")]
with open("/kaggle/working/pilot_generated_edits.jsonl", "w") as f:
    for r in merged:
        f.write(json.dumps(r) + "\n")
print("total generated:", len(merged),
      "| quality pass:", sum(r.get("quality_gate_status") == "pass" for r in merged))
with zipfile.ZipFile("/kaggle/working/diffusion_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/pilot_generated_edits.jsonl", "pilot_generated_edits.jsonl")
    for s in (0, 1):
        gs = f"/kaggle/working/gen_summary_shard{s}.json"
        if os.path.exists(gs):
            z.write(gs, f"gen_summary_shard{s}.json")
        for png in glob.glob(f"/kaggle/working/edits/shard{s}/*.png"):
            z.write(png, os.path.relpath(png, "/kaggle/working"))   # -> edits/shardN/<id>.png
print("DOWNLOAD just diffusion_out.zip — it now contains edits/ + pilot_generated_edits.jsonl + summaries")

## Back on the Mac — run the gate (decides whether VLM may start)
Unzip `diffusion_out.zip` into `data/edits/main_real_200/`, drop
`pilot_generated_edits.jsonl` into `data/results/main_real_200/`, then run
quality + `certvic.validation.edit_detectability` + `tiny_pilot_go_no_go`.
**GO only if AUC < 0.80.** See `docs/runbooks/KAGGLE_T4x2_DIFFUSION_EDITS.md` §7.